In [11]:
from PIL import Image
import subprocess
import os
import tempfile

def compress_image(input_path, output_path, filesize_threshold=500):
    """Function to compress an image using pillow and optionally ffmpeg, 
    to a maximum of the specified filesize threshold [KB]. It will compress 
    in multiple stages if necessary to meet the file size threshold."""

    # open an image with pillow
    with Image.open(input_path) as my_image:

        # original image height and width
        original_image_height = my_image.height
        original_image_width = my_image.width
        original_filesize = len(my_image.fp.read())/1024

        print(f"Original image size: height={original_image_height}, width={original_image_width}")
        print("The original filesize of image is: ", round(original_filesize), "KB")

        # resize the image
        new_image_width = 1200
        my_image = my_image.resize((new_image_width,int(my_image.height * (new_image_width / original_image_width))))

        new_image_height = my_image.height
        
    # save the image to a temporary file based on input file-extension
    if input_path.endswith('.jpg'):
        suffix='.jpg'
    elif input_path.endswith('.png'):
        suffix='.png'
    else:
        raise ValueError("Unknown format")
       
    with tempfile.NamedTemporaryFile(suffix=suffix, delete_on_close=True, dir="D:") as resized_image:
        my_image.save(resized_image)
        resized_image_path = resized_image.name

        # Check new file size after resizing
        new_filesize = os.path.getsize(resized_image_path) / 1024
        print(f"Resized image size: height={new_image_height}, width={new_image_width}")
        print("The filesize of resized image is: ", round(new_filesize), "KB")

        # Done if threshold is met after resizing only, otherwise continue with ffmpeg compression steps
        if not new_filesize > filesize_threshold:
            my_image.save(r"D:\Code\Project_Van_Hool\Media\dak\dak1_resized.jpg")

        else:
            print("Compressing further with ffmpeg...")
            # ffmpeg_command = ["ffmpeg", "-i", resized_image_path, "-y"]
            if input_path.endswith('.jpg'):
                quality_range = range(1, 31, 1)
                encoder = "-q:v"
                # options = ["-update", "1", "-frames:v", "1"]
                options = []
            elif input_path.endswith('.png'):
                quality_range = range(51, 2, -1)
                encoder = "-codec:v"
                # options = ["libx265", "-crf", "-update", "1", "-frames:v", "1"]
                options = ["libx265", "-crf"]
            else:
                raise ValueError("Unknown format")
            
            for quality_value in quality_range:
                # create second temporary file for storing the new compressed image
                with tempfile.NamedTemporaryFile(suffix=suffix, delete_on_close=True, dir="D:") as compressed_image:
                    compressed_image_path = compressed_image.name 

                    # add quality value and temporary filepath to ffmpeg command
                    # ffmpeg_command.append(str(quality_value))
                    # ffmpeg_command.append(compressed_image_path)
                    ffmpeg_command = [
                        "ffmpeg", 
                        "-i",
                        input_path,
                        "-y", 
                        encoder,
                        str(quality_value),
                    ]
                    ffmpeg_command.extend(options)
                    ffmpeg_command.append(compressed_image_path)
                    print(ffmpeg_command)
                    
                    # run ffmpeg using subprocess
                    print(f"Trying quality value: {quality_value} ...")
                    result = subprocess.run(ffmpeg_command, capture_output=True, text=True)
                    print("STDOUT:", result.stdout)
                    print("STDERR:", result.stderr)

                    # Check new file size after compression
                    new_filesize = os.path.getsize(compressed_image_path) / 1024
                    print(f"Quality_value: {quality_value}, filesize: {round(new_filesize)} KB")
                    
                    if 0 < new_filesize <= filesize_threshold:
                        print(f"Target reached with quality_value {quality_value}")
                        # Replace resized image file with compressed version
                        # os.replace(compressed_image_path, output_path)
                        break
                    elif new_filesize == 0:
                        raise ValueError("Output filesize is zero!")
    print("Done")

                
if __name__ == "__main__":

    input_path = r"D:\Code\Project_Van_Hool\Media\dak\dak1.jpg"
    output_path = r"D:\Code\Project_Van_Hool\Media\dak\dak1_compressed.jpg"
    compress_image(input_path, output_path)

Original image size: height=3168, width=1584
The original filesize of image is:  3176 KB
Resized image size: height=2400, width=1200
The filesize of resized image is:  538 KB
Compressing further with ffmpeg...
['ffmpeg', '-i', 'D:\\Code\\Project_Van_Hool\\Media\\dak\\dak1.jpg', '-y', '-q:v', '1', 'd:\\Code\\Project_Van_Hool\\tmp5rtyrxfw.jpg']
Trying quality value: 1 ...
STDOUT: 
STDERR: ffmpeg version 8.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --

ValueError: Output filesize is zero!